# TP1 - Etude d'un noyau temps réel (FreeRTOS)
## Gestion des tâches

**ECUE : Atelier Systèmes Temps Réel**  
**Support matériel : Arduino AVR + FreeRTOS**

Ce notebook accompagne le TP1. Les programmes Arduino/FreeRTOS se compilent et se téléversent depuis l'environnement Arduino. Les cellules Python servent à observer et à vérifier les concepts de manière reproductible dans Jupyter.

## Objectifs

À la fin du TP, vous devez être capable de :

- définir une tâche avec le noyau FreeRTOS ;
- créer une tâche avec `xTaskCreate()` ;
- créer des tâches périodiques avec `vTaskDelay()` et `vTaskDelayUntil()` ;
- supprimer, suspendre et reprendre une tâche ;
- distinguer les états **Ready**, **Running**, **Blocked** et **Suspended** ;
- modifier et lire la priorité avec `vTaskPrioritySet()` et `uxTaskPriorityGet()`.

## Rappels : états d'une tâche

- **Ready** : la tâche peut s'exécuter et attend le processeur.
- **Running** : la tâche utilise actuellement le processeur.
- **Blocked** : la tâche attend un délai ou un événement.
- **Suspended** : la tâche est explicitement suspendue et ne peut pas s'exécuter.

Une tâche FreeRTOS est généralement une fonction sans valeur de retour contenant une boucle infinie.

## Protocole de manipulation

Avant chaque essai Arduino :

1. vérifier la carte et le port dans **Outils** ;
2. compiler puis téléverser le programme ;
3. ouvrir le moniteur série à **9600 bauds** ;
4. ne modifier qu'un paramètre à la fois ;
5. noter la priorité, le délai, la période et l'ordre des messages observés.

Les simulations Python illustrent le comportement logique. Elles ne remplacent pas la mesure sur la carte : le temps d'affichage série, la configuration du tick et le port AVR influencent les résultats réels.

## Exercice 1 - Création de tâches

Dans Arduino, ouvrir `FreeRTOS_AVR/FreeRTOSBook/Example001`, vérifier la carte et le port, téléverser le programme puis observer le moniteur série. Deux tâches de même priorité écrivent successivement leur nom.

La création suit cette forme :

```cpp
xTaskCreate(vTask1, "Task 1", 200, NULL, 1, NULL);
xTaskCreate(vTask2, "Task 2", 200, NULL, 1, NULL);
vTaskStartScheduler();
```

Référence : [`Example001.ino`](FreeRTOS_AVR/examples/FreeRTOSBook/Example001/Example001.ino).

## Exercice 2 - Création de tâches avec paramètres

Ouvrir `Example002`. Une même fonction peut être utilisée par plusieurs tâches ; le paramètre `pvParameters` permet de différencier leur comportement.

```cpp
const char *message1 = "Task 1 is running\r\n";
const char *message2 = "Task 2 is running\r\n";
xTaskCreate(vTaskFunction, "Task 1", 200, (void *)message1, 1, NULL);
xTaskCreate(vTaskFunction, "Task 2", 200, (void *)message2, 1, NULL);
```

**Travail demandé :** ajouter une troisième tâche avec le même comportement, puis étudier l'effet de l'ordre de création.

In [ ]:
from dataclasses import dataclass

@dataclass
class Tache:
    nom: str
    priorite: int

def tourniquet(taches, tours=2):
    """Approximation pédagogique du partage entre tâches de même priorité."""
    return [tache.nom for _ in range(tours) for tache in taches]

taches = [Tache('Task 1', 1), Tache('Task 2', 1), Tache('Task 3', 1)]
tourniquet(taches)

## Exercice 3 - Priorité d'une tâche

Ouvrir `Example003`, modifier les priorités puis expliquer quelle tâche s'exécute en premier. Dans FreeRTOS, la tâche prête de plus haute priorité est choisie. À priorité égale, l'ordonnanceur peut partager le temps selon la configuration.

## Exercice 4 - `vTaskDelay()`

Ouvrir `Example004`. L'appel `vTaskDelay(n)` place la tâche appelante dans l'état **Blocked** pendant `n` ticks.

```cpp
vTaskDelay(250 / portTICK_PERIOD_MS);
```

Modifier ensuite l'appel en `vTaskDelay(5)` et expliquer l'effet observé. Estimer le nombre de ticks consommés par l'affichage série.

In [ ]:
def simuler_ordonnanceur(taches, ticks=12):
    """Simule le choix preemptif d'une tache sur une suite de ticks.

    Chaque tache est copiee afin de conserver les parametres d'origine.
    A priorite egale, les taches pretes sont servies a tour de role.
    """
    etat = [dict(tache) for tache in taches]
    journal = []
    index_egalite = 0

    for tick in range(ticks):
        pretes = [tache for tache in etat if tache['reveil'] <= tick]
        if not pretes:
            journal.append((tick, 'IDLE'))
            continue

        priorite_max = max(tache['priorite'] for tache in pretes)
        candidats = [tache for tache in pretes if tache['priorite'] == priorite_max]
        choisie = candidats[index_egalite % len(candidats)]
        index_egalite += 1
        journal.append((tick, choisie['nom'], choisie['priorite']))
        choisie['reveil'] = tick + choisie['delai']

    return journal

configuration = [
    {'nom': 'Task 1', 'priorite': 1, 'delai': 3, 'reveil': 0},
    {'nom': 'Task 2', 'priorite': 2, 'delai': 5, 'reveil': 0},
]
simuler_ordonnanceur(configuration), configuration

## Exercice 5 - `vTaskDelayUntil()`

Ouvrir `Example005`. Contrairement à `vTaskDelay()`, `vTaskDelayUntil()` utilise une échéance absolue et maintient une période régulière même si le temps d'exécution varie.

```cpp
TickType_t derniere_reveil = xTaskGetTickCount();
vTaskDelayUntil(&derniere_reveil, 250 / portTICK_PERIOD_MS);
```

**Question :** expliquer la différence entre un délai relatif (`vTaskDelay`) et une période absolue (`vTaskDelayUntil`).

In [ ]:
periode = 5
nombre_d_executions = 5
reveils = [i * periode for i in range(nombre_d_executions)]
print('Réveils avec vTaskDelayUntil :', reveils)

# Avec un délai relatif, le temps de travail se cumule à chaque période.
temps_travail = 1
reveils_relatifs = [i * (periode + temps_travail) for i in range(nombre_d_executions)]
print('Réveils avec vTaskDelay :    ', reveils_relatifs)

In [ ]:
def comparer_delais(periode, temps_travail, executions=5):
    """Compare les instants de reveil relatifs et absolus."""
    relatifs = []
    absolus = []
    instant_relatif = 0
    prochaine_echeance = 0

    for _ in range(executions):
        relatifs.append(instant_relatif)
        absolus.append(prochaine_echeance)
        instant_relatif += temps_travail + periode
        prochaine_echeance += periode

    return relatifs, absolus

reveils_relatifs, reveils_absolus = comparer_delais(5, 2)
print('vTaskDelay     :', reveils_relatifs)
print('vTaskDelayUntil:', reveils_absolus)
print('Derive finale  :', reveils_relatifs[-1] - reveils_absolus[-1], 'ticks')

## Exercice 6 - Tâches bloquantes et non bloquantes

Ouvrir `Example006`, observer les tâches `Task1`, `Task2` et `Periodic`, puis modifier la période de la tâche périodique à **10 ms**, **50 ms** et **150 ms**.

Une tâche bloquante appelle une primitive qui la retire temporairement de l'état **Ready**. Une tâche non bloquante continue à consommer du temps processeur tant qu'elle ne cède pas la main.

Décrire l'effet sur l'occupation du processeur, la régularité des réveils et les éventuels retards. Une tâche périodique doit disposer d'un temps d'exécution compatible avec sa période. Comparer l'utilisation totale à 100 % : une charge supérieure à 100 % indique que les tâches ne peuvent pas toutes respecter leurs échéances dans ce modèle simplifié.

In [ ]:
def charge_utilisation(taches):
    return sum(tache['execution_ms'] / tache['periode_ms'] for tache in taches)

taches = [
    {'nom': 'Periodic', 'execution_ms': 3, 'periode_ms': 10},
    {'nom': 'Sensor', 'execution_ms': 8, 'periode_ms': 50},
]
for periode in (10, 50, 150):
    taches[0]['periode_ms'] = periode
    print(f'Période={periode:>3} ms -> utilisation={charge_utilisation(taches):.1%}')

In [ ]:
def verifier_charge(taches):
    utilisation = charge_utilisation(taches)
    return {
        'utilisation': utilisation,
        'echeances_potentiellement_tenables': utilisation <= 1.0,
        'marge_cpu': max(0.0, 1.0 - utilisation),
    }

for periode in (10, 50, 150):
    scenario = [
        {'nom': 'Periodic', 'execution_ms': 3, 'periode_ms': periode},
        {'nom': 'Sensor', 'execution_ms': 8, 'periode_ms': 50},
    ]
    print(f'Période={periode:>3} ms -> {verifier_charge(scenario)}')

## API complémentaires utiles

Ces API permettent d'observer le noyau et de mieux diagnostiquer une application temps réel.

### 1. Observer le temps et l'état du scheduler

```cpp
TickType_t maintenant = xTaskGetTickCount();
TickType_t depuis_isr = xTaskGetTickCountFromISR();
BaseType_t etat = xTaskGetSchedulerState();
```

`xTaskGetTickCount()` est utilisable dans une tâche ; la variante `FromISR` est réservée aux interruptions. L'état retourné par `xTaskGetSchedulerState()` permet de distinguer un scheduler non démarré, en fonctionnement ou suspendu.

### 2. Observer les tâches

```cpp
UBaseType_t nombre = uxTaskGetNumberOfTasks();
TaskHandle_t courante = xTaskGetCurrentTaskHandle();
UBaseType_t priorite = uxTaskPriorityGet(courante);
```

Ces fonctions sont utiles pour instrumenter le système, mais l'affichage série doit rester ponctuel : il perturbe les mesures temporelles.

### 3. Céder explicitement le processeur

```cpp
taskYIELD();
```

`taskYIELD()` demande une nouvelle sélection de tâche. Elle ne bloque pas la tâche appelante et ne garantit pas qu'une autre tâche sera choisie.

### 4. Suspendre temporairement l'ordonnanceur

```cpp
vTaskSuspendAll();
// section très courte sans changement de contexte
xTaskResumeAll();
```

Cette paire ne doit pas remplacer un mutex : elle suspend la commutation des tâches et doit rester extrêmement courte. Elle ne désactive pas nécessairement les interruptions. Ne l'utilisez jamais autour d'une attente ou d'une opération lente.

**Configuration :** vérifier les options `INCLUDE_xTaskGetSchedulerState`, `INCLUDE_xTaskGetCurrentTaskHandle`, `INCLUDE_uxTaskGetNumberOfTasks` et `INCLUDE_xTaskGetTickCount` dans `FreeRTOSConfig.h` selon la version du noyau.

In [ ]:
def observer_noyau(ticks, scheduler='RUNNING', taches=3):
    """Modele minimal de l'instrumentation d'un noyau FreeRTOS."""
    return {
        'tick': ticks,
        'scheduler': scheduler,
        'taches_gerees': taches,
        'instrumentation_autorisee': scheduler == 'RUNNING',
    }

for tick in (0, 10, 20):
    print(observer_noyau(tick))

## Exercice 7 - Idle Task et Idle Task Hook

Ouvrir `Example007`. La tâche idle s'exécute lorsque aucune tâche applicative prête ne peut utiliser le processeur. Utiliser `uxTaskGetNumberOfTasks()` pour afficher le nombre de tâches gérées par le noyau.

```cpp
UBaseType_t nombre = uxTaskGetNumberOfTasks();
Serial.println(nombre);
```

**Question :** expliquer pourquoi une tâche applicative qui monopolise le processeur peut empêcher la tâche idle de s'exécuter.

## Exercice 8 - Modification dynamique des priorités

Ouvrir `Example008`, puis afficher les priorités courantes des tâches avec :

```cpp
UBaseType_t priorite = uxTaskPriorityGet(xTask);
vTaskPrioritySet(xTask, nouvelle_priorite);
```

Vérifier que `INCLUDE_vTaskPriorityGet` et `INCLUDE_vTaskPrioritySet` valent `1` dans la configuration FreeRTOS. Expliquer le changement de comportement après la modification.

## Exercice 9 - Suppression et suspension des tâches

Ouvrir `Example009`. `vTaskDelete()` retire une tâche de la gestion du noyau ; la tâche idle récupère ensuite la mémoire associée.

```cpp
vTaskDelete(xTask);
vTaskSuspend(xTask);
vTaskResume(xTask);
```

Modifier le programme pour obtenir un comportement équivalent avec `vTaskSuspend()` et `vTaskResume()`. Comparer les états et le cycle de vie des deux solutions.

In [ ]:
transitions = {
    'Ready -> Running': 'le planificateur sélectionne la tâche',
    'Running -> Blocked': 'vTaskDelay() ou attente d’un événement',
    'Blocked -> Ready': 'le délai expire ou l’événement arrive',
    'Running -> Suspended': 'vTaskSuspend() est appelée',
    'Suspended -> Ready': 'vTaskResume() est appelée',
    'Running -> Deleted': 'vTaskDelete() est appelée',
}
for transition, cause in transitions.items():
    print(f'{transition}: {cause}')

## Compte rendu et critères de réussite

Pour chaque exercice, joindre :

1. les observations du moniteur série ;
2. le programme modifié et les paramètres testés ;
3. une explication des états et des décisions de l'ordonnanceur ;
4. les mesures de période, de délai ou d'utilisation lorsque cela est demandé ;
5. une comparaison entre le résultat attendu et le résultat observé.

### Checklist

- [ ] Les deux tâches de l'exercice 1 sont créées avant `vTaskStartScheduler()`.
- [ ] Le paramètre `pvParameters` est différent pour les instances de l'exercice 2.
- [ ] La tâche de plus haute priorité est identifiée et son comportement est expliqué.
- [ ] La différence entre délai relatif et échéance absolue est démontrée.
- [ ] La charge CPU est calculée et interprétée.
- [ ] Les états **Ready**, **Running**, **Blocked**, **Suspended** et **Deleted** sont distingués.
- [ ] Les API utilisées sont comparées à leur observation matérielle.

Les exemples Arduino complets sont disponibles dans `FreeRTOS_AVR/examples/FreeRTOSBook/`.